In [1]:
import time
import math
import requests
import pandas as pd
import numpy as np
from scipy.optimize import curve_fit
from datetime import datetime, timezone, timedelta
from bot_template import BaseBot, OrderBook, OrderRequest, Side, Trade

In [2]:
LONDON_LAT, LONDON_LON = 51.5074, -0.1278
THAMES_MEASURE = "0006-level-tidal_level-i-15_min-mAOD"

In [4]:
def get_weather(past_steps=96, forecast_steps=96):
    """15-min weather for London. 96 steps = 24 hours.

    Returns DataFrame with: time, temperature, wind_speed, humidity,
    precipitation, cloud_cover, visibility, apparent_temperature.
    """
    variables = "temperature_2m,apparent_temperature,relative_humidity_2m,precipitation,wind_speed_10m,cloud_cover,visibility"
    resp = requests.get("https://api.open-meteo.com/v1/forecast", params={
        "latitude": self.LONDON_LAT, "longitude": self.LONDON_LON,
        "minutely_15": variables,
        "past_minutely_15": past_steps,
        "forecast_minutely_15": forecast_steps,
        "timezone": "Europe/London",
    })
    resp.raise_for_status()
    m = resp.json()["minutely_15"]
    return pd.DataFrame({
        "time": pd.to_datetime(m["time"]).tz_localize("Europe/London"),
        "temperature": m["temperature_2m"],
        "apparent_temperature": m["apparent_temperature"],
        "humidity": m["relative_humidity_2m"],
        "precipitation": m["precipitation"],
        "wind_speed": m["wind_speed_10m"],
        "cloud_cover": m["cloud_cover"],
        "visibility": m["visibility"],
    })

def celsius_to_fahrenheit(c: float) -> float:
    """Convert Open-Meteo Celsius to Fahrenheit for settlement."""
    return (c * 9/5) + 32

In [ ]:
df = get_weather(96, 96)
theos = {}

# 2. WX_SPOT: Target Sunday 12:00 PM specifically
target_time = pd.Timestamp("2026-03-01 12:00:00", tz="Europe/London")

# Find the row closest to our target settlement time
settlement_row = df.iloc[(df['time'] - target_time).abs().argsort()[:1]]

if not settlement_row.empty:
    temp_c = settlement_row['temperature'].values[0]
    humidity = settlement_row['humidity'].values[0]
    temp_f = celsius_to_fahrenheit(temp_c)
    
    # Settlement formula: temp_F * humidity_%
    theos["WX_SPOT"] = temp_f * humidity

# 3. WX_SUM: Sum of (temp_F * humidity_%) / 100 over the 24h session
# Define the 24h session window (e.g., Saturday 12pm to Sunday 12pm)
session_start = target_time - pd.Timedelta(hours=24)
session_df = df[(df['time'] >= session_start) & (df['time'] <= target_time)].copy()

if not session_df.empty:
    # Apply formula to each 15-min interval in the session
    session_df['interval_val'] = session_df.apply(
        lambda row: celsius_to_fahrenheit(row['temperature']) * row['humidity'], 
        axis=1
    )
    theos["WX_SUM"] = session_df['interval_val'].sum() / 100.0